# Sampling Techniques: Simple Random, Stratified and Cluster
Works on **any dataset** (CSV / Excel). A file explorer opens so you can pick the file, then you choose the columns.

| Method | Idea | Best when |
|---|---|---|
| Simple Random | Every unit has equal chance | Population is homogeneous |
| Systematic | Every k-th unit after random start | Ordered list, no hidden periodicity |
| Stratified | Split into homogeneous groups (strata), sample from **every** stratum | Groups differ a lot from each other |
| Cluster | Split into heterogeneous groups (clusters), randomly pick **whole** clusters | Population is spread out, cost is high |

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

## 2. Select Dataset from File Explorer

In [ ]:
import tkinter as tk
from tkinter import filedialog

root = tk.Tk()
root.withdraw()
root.attributes('-topmost', True)
file_path = filedialog.askopenfilename(
    title="Select your dataset",
    filetypes=[("Data files", "*.csv *.xlsx *.xls *.txt *.tsv"), ("All files", "*.*")]
)
root.destroy()

if not file_path:
    raise SystemExit("No file selected. Re-run this cell.")

print("Selected file:", file_path)

if file_path.endswith('.csv'):
    df = pd.read_csv(file_path)
elif file_path.endswith(('.xlsx', '.xls')):
    df = pd.read_excel(file_path)
else:
    df = pd.read_csv(file_path, sep=None, engine='python')

df = df.reset_index(drop=True)
print("Shape:", df.shape)
df.head()

## 3. Choose Columns

In [ ]:
print("All columns:")
for i, c in enumerate(df.columns):
    print(f"{i}: {c}  ({df[c].dtype}, {df[c].nunique()} unique)")

def pick(prompt):
    v = input(prompt).strip()
    return df.columns[int(v)] if v.isdigit() and v not in df.columns else v

value_col   = pick("\nNumeric VARIABLE to estimate (name or index): ")
strata_col  = pick("STRATA column - categorical, e.g. gender/region (name or index): ")
cluster_col = pick("CLUSTER column - groups of units, e.g. city/school (name or index): ")

df = df.dropna(subset=[value_col, strata_col, cluster_col]).reset_index(drop=True)
df[value_col] = pd.to_numeric(df[value_col], errors='coerce')
df = df.dropna(subset=[value_col]).reset_index(drop=True)

# If the strata column is numeric with many values, bin it into 4 groups
if pd.api.types.is_numeric_dtype(df[strata_col]) and df[strata_col].nunique() > 10:
    df[strata_col] = pd.qcut(df[strata_col], 4, duplicates='drop').astype(str)
    print("Strata column was numeric -> binned into quartiles.")

N = len(df)
pop_mean = df[value_col].mean()
pop_std = df[value_col].std(ddof=0)
print(f"\nPopulation size N = {N}")
print(f"True population mean of '{value_col}' = {pop_mean:.4f}")
print(f"Number of strata   : {df[strata_col].nunique()}")
print(f"Number of clusters : {df[cluster_col].nunique()}")

## 4. Sample Size

In [ ]:
n = int(input(f"Enter sample size n (population N = {N}): "))
n = min(n, N)
print("Sample size n =", n, f"({n/N*100:.1f}% of population)")

## 5. Simple Random Sampling (SRS)
Every unit has the same probability `n/N` of being selected.

In [ ]:
# Without replacement (SRSWOR)
srs = df.sample(n=n, replace=False, random_state=1)
print(f"SRSWOR mean = {srs[value_col].mean():.4f}   (true = {pop_mean:.4f})")

# With replacement (SRSWR)
srs_wr = df.sample(n=n, replace=True, random_state=1)
print(f"SRSWR  mean = {srs_wr[value_col].mean():.4f}")

srs.head()

## 6. Systematic Sampling (bonus)
Pick a random start, then every k-th unit where `k = N/n`.

In [ ]:
k = max(N // n, 1)
start = np.random.randint(0, k)
sys_sample = df.iloc[start::k].head(n)
print(f"Interval k = {k}, random start = {start}, sample size = {len(sys_sample)}")
print(f"Systematic mean = {sys_sample[value_col].mean():.4f}")

## 7. Stratified Sampling
Divide the population into strata, sample within **each** stratum. Estimator: $\bar{y}_{st} = \sum_h W_h \bar{y}_h$ where $W_h = N_h/N$.

Three allocations:
- **Proportional:** $n_h = n \cdot N_h/N$
- **Equal:** $n_h = n/H$
- **Neyman (optimal):** $n_h = n \cdot \dfrac{N_h S_h}{\sum N_h S_h}$ (larger and more variable strata get more)

In [ ]:
strata_info = df.groupby(strata_col)[value_col].agg(N_h='count', mean_h='mean', std_h=lambda x: x.std(ddof=1)).fillna(0)
strata_info['W_h'] = strata_info['N_h'] / N
H = len(strata_info)

def allocate(method):
    if method == 'proportional':
        nh = strata_info['W_h'] * n
    elif method == 'equal':
        nh = pd.Series(n / H, index=strata_info.index)
    elif method == 'neyman':
        w = strata_info['N_h'] * strata_info['std_h']
        nh = n * w / w.sum() if w.sum() > 0 else strata_info['W_h'] * n
    nh = np.round(nh).astype(int).clip(lower=1)
    return np.minimum(nh, strata_info['N_h'].astype(int))   # cannot exceed stratum size

def stratified_sample(method='proportional', seed=None):
    nh = allocate(method)
    parts = [g.sample(n=nh[name], random_state=seed) for name, g in df.groupby(strata_col)]
    return pd.concat(parts), nh

def stratified_mean(sample):
    # weighted by population stratum weights W_h
    means = sample.groupby(strata_col)[value_col].mean()
    return (means * strata_info['W_h']).sum()

strat_results = {}
alloc_table = pd.DataFrame(index=strata_info.index)
alloc_table['N_h'] = strata_info['N_h']
for m in ['proportional', 'equal', 'neyman']:
    s, nh = stratified_sample(m, seed=1)
    alloc_table[f'n_h ({m})'] = nh
    strat_results[m] = stratified_mean(s)
    print(f"Stratified ({m:12s}) mean = {strat_results[m]:.4f}   (true = {pop_mean:.4f})")

alloc_table

## 8. Cluster Sampling
Randomly select **whole clusters** and survey every unit inside them. Cheaper, but units within a cluster are similar, so precision is usually lower than SRS.

- **One-stage:** take all units in chosen clusters
- **Two-stage:** pick clusters, then randomly sample units inside each chosen cluster

In [ ]:
clusters = df[cluster_col].unique()
M = len(clusters)
avg_cluster_size = N / M
k_clusters = max(int(round(n / avg_cluster_size)), 1)     # number of clusters to reach about n units
k_clusters = min(k_clusters, M)
print(f"Total clusters M = {M}, avg cluster size = {avg_cluster_size:.1f}")
print(f"Clusters selected k = {k_clusters}")

# One-stage cluster sample
chosen = np.random.choice(clusters, size=k_clusters, replace=False)
cluster_sample = df[df[cluster_col].isin(chosen)]
print(f"\nOne-stage cluster sample size = {len(cluster_sample)}")
print(f"One-stage cluster mean = {cluster_sample[value_col].mean():.4f}   (true = {pop_mean:.4f})")
print("Chosen clusters:", list(chosen)[:10], '...' if len(chosen) > 10 else '')

# Two-stage cluster sample: pick clusters then 50% of units in each
two_stage = (df[df[cluster_col].isin(chosen)]
             .groupby(cluster_col, group_keys=False)
             .apply(lambda g: g.sample(frac=0.5, random_state=1) if len(g) > 1 else g))
print(f"\nTwo-stage cluster sample size = {len(two_stage)}")
print(f"Two-stage cluster mean = {two_stage[value_col].mean():.4f}")

## 9. Compare Single Samples with True Mean

In [ ]:
single = {
    'Simple Random': srs[value_col].mean(),
    'Systematic': sys_sample[value_col].mean(),
    'Stratified (prop.)': strat_results['proportional'],
    'Stratified (equal)': strat_results['equal'],
    'Stratified (Neyman)': strat_results['neyman'],
    'Cluster (1-stage)': cluster_sample[value_col].mean(),
    'Cluster (2-stage)': two_stage[value_col].mean(),
}
cmp_df = pd.DataFrame({'Estimate': single})
cmp_df['True mean'] = pop_mean
cmp_df['Error'] = cmp_df['Estimate'] - pop_mean
cmp_df['% Error'] = cmp_df['Error'] / pop_mean * 100
display(cmp_df.round(4))

plt.figure(figsize=(10, 4))
plt.barh(cmp_df.index, cmp_df['Estimate'], color='skyblue')
plt.axvline(pop_mean, color='r', ls='--', label='True mean')
plt.xlim(cmp_df['Estimate'].min()*0.95, cmp_df['Estimate'].max()*1.05)
plt.legend(); plt.title('Estimates from one sample of each method'); plt.show()

## 10. Repeated Sampling Simulation (bias and variance of each method)
One sample can be lucky. Repeating many times shows which method is more **precise** (lower standard error).

In [ ]:
n_sims = 1000
sim = {name: [] for name in ['Simple Random', 'Systematic', 'Stratified (prop.)',
                             'Stratified (Neyman)', 'Cluster (1-stage)']}

for _ in range(n_sims):
    sim['Simple Random'].append(df[value_col].sample(n=n).mean())

    st = np.random.randint(0, k)
    sim['Systematic'].append(df.iloc[st::k].head(n)[value_col].mean())

    s, _ = stratified_sample('proportional')
    sim['Stratified (prop.)'].append(stratified_mean(s))

    s, _ = stratified_sample('neyman')
    sim['Stratified (Neyman)'].append(stratified_mean(s))

    ch = np.random.choice(clusters, size=k_clusters, replace=False)
    sim['Cluster (1-stage)'].append(df[df[cluster_col].isin(ch)][value_col].mean())

sim_df = pd.DataFrame(sim)

summary = pd.DataFrame({
    'Mean of estimates': sim_df.mean(),
    'Bias': sim_df.mean() - pop_mean,
    'Std Error': sim_df.std(),
    'RMSE': np.sqrt(((sim_df - pop_mean)**2).mean())
})
summary['Relative efficiency vs SRS'] = (summary.loc['Simple Random', 'Std Error'] / summary['Std Error'])**2
display(summary.round(4))
print("Relative efficiency > 1 means the method is MORE precise than simple random sampling.")

## 11. Sampling Distributions of Each Method

In [ ]:
plt.figure(figsize=(11, 5))
for c in sim_df.columns:
    sns.kdeplot(sim_df[c], label=c, fill=True, alpha=.15)
plt.axvline(pop_mean, color='k', ls='--', label='True mean')
plt.title(f'Sampling distribution of the mean (n={n}, {n_sims} simulations)')
plt.legend(); plt.show()

plt.figure(figsize=(9, 5))
sim_df.boxplot()
plt.axhline(pop_mean, color='r', ls='--')
plt.xticks(rotation=20); plt.title('Spread of estimates by method'); plt.show()

## 12. Visual Check: Does the Sample Represent the Population?

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))

# Strata proportions: population vs samples
s_prop, _ = stratified_sample('proportional', seed=3)
props = pd.DataFrame({
    'Population': df[strata_col].value_counts(normalize=True),
    'SRS': srs[strata_col].value_counts(normalize=True),
    'Stratified': s_prop[strata_col].value_counts(normalize=True),
    'Cluster': cluster_sample[strata_col].value_counts(normalize=True),
}).fillna(0)
props.plot(kind='bar', ax=ax[0]); ax[0].set_title(f'Proportion of each stratum ({strata_col})')
ax[0].set_ylabel('Proportion'); ax[0].tick_params(axis='x', rotation=30)

# Distribution of the variable
sns.kdeplot(df[value_col], ax=ax[1], label='Population', lw=3, color='k')
sns.kdeplot(srs[value_col], ax=ax[1], label='SRS')
sns.kdeplot(s_prop[value_col], ax=ax[1], label='Stratified')
sns.kdeplot(cluster_sample[value_col], ax=ax[1], label='Cluster')
ax[1].set_title(f'Distribution of {value_col}'); ax[1].legend()
plt.tight_layout(); plt.show()

## 13. Standard Error and Confidence Interval from a Single Sample

In [ ]:
# SRS: SE with finite population correction (fpc)
fpc = 1 - n / N
se_srs = srs[value_col].std(ddof=1) / np.sqrt(n) * np.sqrt(fpc)
ci_srs = (srs[value_col].mean() - 1.96*se_srs, srs[value_col].mean() + 1.96*se_srs)
print(f"SRS         : mean={srs[value_col].mean():.3f}, SE={se_srs:.4f}, 95% CI=({ci_srs[0]:.3f}, {ci_srs[1]:.3f})")

# Stratified: Var = sum W_h^2 * (1 - n_h/N_h) * s_h^2 / n_h
s_prop, nh = stratified_sample('proportional', seed=1)
sh2 = s_prop.groupby(strata_col)[value_col].var(ddof=1).fillna(0)
var_st = (strata_info['W_h']**2 * (1 - nh/strata_info['N_h']) * sh2 / nh).sum()
se_st = np.sqrt(var_st)
m_st = stratified_mean(s_prop)
print(f"Stratified  : mean={m_st:.3f}, SE={se_st:.4f}, 95% CI=({m_st-1.96*se_st:.3f}, {m_st+1.96*se_st:.3f})")

# Cluster: use cluster means (equal-size approximation)
cm = cluster_sample.groupby(cluster_col)[value_col].mean()
se_cl = cm.std(ddof=1) / np.sqrt(len(cm)) * np.sqrt(1 - len(cm)/M) if len(cm) > 1 else np.nan
m_cl = cluster_sample[value_col].mean()
print(f"Cluster     : mean={m_cl:.3f}, SE={se_cl:.4f}, 95% CI=({m_cl-1.96*se_cl:.3f}, {m_cl+1.96*se_cl:.3f})")

## Key Points (for viva)
| | Simple Random | Stratified | Cluster |
|---|---|---|---|
| Units grouped by | none | **Homogeneous within**, different between strata | **Heterogeneous within**, similar between clusters |
| What is sampled | Individual units | Units from **every** stratum | **Whole clusters** (some chosen) |
| Precision | baseline | Usually **higher** than SRS | Usually **lower** than SRS (design effect > 1) |
| Cost | medium-high | medium | **lowest** |
| Needs | Complete sampling frame | Strata info for every unit | List of clusters only |

- **SRSWOR** has no repeats, **SRSWR** can repeat units.
- **Proportional allocation:** $n_h = n N_h / N$. **Neyman allocation** gives the minimum variance for a fixed total n.
- **Finite population correction:** $(1 - n/N)$ reduces variance when sampling a large fraction of the population.
- **Stratified vs Cluster:** in stratified you sample *from all groups*; in cluster you sample *whole groups* (only some of them).
- **Sampling frame** = list of all units from which the sample is drawn. **Sampling error** is the difference between a sample estimate and the true parameter.